# 17.1 马尔可夫决策过程与贝尔曼方程 / MDP & Bellman Equations

**中文**：前面所有的机器学习都是"从固定数据里学"——有一批标注好的样本，学个映射。但**强化学习(Reinforcement Learning, RL)** 完全不同:一个**智能体(agent)** 在**环境(environment)** 里**行动**、观察**反馈**、不断**试错**，目标是**最大化长期累积奖励**。没有人告诉它"正确答案是什么"，只有奖励信号。下棋、打游戏、机器人控制、推荐、自动驾驶、对齐大模型(RLHF)——都是 RL。本部分(Part 17)从数学地基一路搭到 PPO。
**English**: All prior ML "learns from a fixed dataset" — a batch of labeled samples, learn a mapping. **Reinforcement Learning (RL)** is entirely different: an **agent** **acts** in an **environment**, observes **feedback**, and **learns by trial and error** to **maximize long-term cumulative reward**. No one gives it "the right answer" — only a reward signal. Games, robotics, recommendation, self-driving, aligning LLMs (RLHF) — all are RL. This part (Part 17) builds from the mathematical foundation up to PPO.

---

**中文**：RL 的数学框架是**马尔可夫决策过程(Markov Decision Process, MDP)**，由五元组 $(S,A,P,R,\gamma)$ 定义:
**English**: RL's mathematical framework is the **Markov Decision Process (MDP)**, a 5-tuple $(S,A,P,R,\gamma)$:
- **状态 $S$ (states)**:环境的所有可能情形(如棋盘布局、机器人位置)。
  **States $S$**: all possible situations (board layout, robot position).
- **动作 $A$ (actions)**:智能体在每个状态能做的选择(上/下/左/右)。
  **Actions $A$**: choices available in each state (up/down/left/right).
- **转移概率 $P(s'|s,a)$**:在状态 $s$ 做动作 $a$ 后转到 $s'$ 的概率(环境的"物理规律")。
  **Transition $P(s'|s,a)$**: probability of moving to $s'$ after action $a$ in $s$ (the environment's "physics").
- **奖励 $R(s,a)$**:做了动作后立即得到的数值反馈。
  **Reward $R(s,a)$**: the immediate numeric feedback after an action.
- **折扣因子 $\gamma\in[0,1]$**:未来奖励的"贴现率"——$\gamma$ 越小越"短视",越大越"远见"。
  **Discount $\gamma\in[0,1]$**: how much future rewards are discounted — smaller = short-sighted, larger = far-sighted.

**中文**：**马尔可夫性质**:下一状态只取决于**当前**状态和动作，与更早的历史无关("无记忆")。这是 MDP 的核心假设，让问题可解。
**English**: The **Markov property**: the next state depends only on the **current** state and action, not on earlier history ("memoryless"). This is the core MDP assumption that makes the problem tractable.

**中文**：智能体的行为由**策略(policy) $\pi(a|s)$** 决定——在状态 $s$ 选各动作的概率。目标是找**最优策略**，使**回报(return)** 最大。回报是折扣累积奖励:$G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2 R_{t+3}+\cdots$
**English**: The agent's behavior is a **policy $\pi(a|s)$** — the probability of each action in state $s$. The goal is the **optimal policy** maximizing the **return** — the discounted cumulative reward: $G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2 R_{t+3}+\cdots$

> 💡 **面试速查 / Interview cheat-sheet（★★★ RL 地基必考）**
> **中文**：RL=智能体在 MDP 里试错最大化长期回报。**MDP=(S,A,P,R,γ)**, 核心是**马尔可夫性**(下一步只依赖当前状态)。**策略 π(a|s)**=行为; **回报 G**=折扣累积奖励; **价值函数**衡量"从某状态出发有多好"——状态价值 $V^\pi(s)=E[G_t|s]$、动作价值 $Q^\pi(s,a)=E[G_t|s,a]$。**贝尔曼方程**是 RL 的心脏:$V^\pi(s)=\sum_a\pi(a|s)\sum_{s'}P(s'|s,a)[R+\gamma V^\pi(s')]$——把"当前价值"递归拆成"即时奖励+下一状态价值"。区分:**有模型(known P,R→动态规划)** vs **无模型(未知→MC/TD 采样学)**。
> **English**: RL = an agent in an MDP learning by trial-and-error to maximize long-term return. **MDP=(S,A,P,R,γ)**, centered on the **Markov property** (next step depends only on the current state). **Policy π(a|s)** = behavior; **return G** = discounted cumulative reward; **value functions** measure "how good is a state" — state value $V^\pi(s)=E[G_t|s]$, action value $Q^\pi(s,a)=E[G_t|s,a]$. The **Bellman equation** is RL's heart: $V^\pi(s)=\sum_a\pi(a|s)\sum_{s'}P(s'|s,a)[R+\gamma V^\pi(s')]$ — recursively splitting "current value" into "immediate reward + next-state value." Distinguish **model-based** (known P,R → dynamic programming) vs **model-free** (unknown → learn by sampling: MC/TD).


In [ ]:

# ============================================================
# 环境:从零实现 GridWorld / GridWorld from scratch
# 中文:一个 4x4 网格。智能体从某格出发, 每步上下左右移动, 目标是到达终点(右下角)。
#      每走一步奖励 -1 (鼓励尽快到达); 到终点结束。撞墙则原地不动。这是 RL 的"Hello World"。
# English: a 4x4 grid. The agent moves up/down/left/right; the goal is the bottom-right cell.
#      Each step gives reward -1 (encouraging speed); reaching the goal ends the episode. Walls = stay put.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)

class GridWorld:
    def __init__(s, size=4):
        s.size=size; s.nS=size*size; s.nA=4                   # 状态数 / 动作数(上右下左)
        s.goal=s.nS-1                                          # 终点=右下角 / goal = bottom-right
        s.A=[(-1,0),(0,1),(1,0),(0,-1)]                       # 4 个动作的 (行,列) 位移 / action deltas
        s.names=["↑","→","↓","←"]
    def rc(s, state): return divmod(state, s.size)            # 状态编号 -> (行,列) / to (row,col)
    def idx(s, r, c): return r*s.size+c                       # (行,列) -> 状态编号 / to index
    def step(s, state, a):
        """返回 (下一状态, 奖励, 是否终止) / returns (next_state, reward, done)."""
        if state==s.goal: return state, 0.0, True             # 已在终点 / terminal
        r,c=s.rc(state); dr,dc=s.A[a]
        nr,nc=r+dr, c+dc
        if 0<=nr<s.size and 0<=nc<s.size: ns=s.idx(nr,nc)     # 合法移动 / valid move
        else: ns=state                                        # 撞墙原地不动 / bump into wall -> stay
        done = (ns==s.goal)
        return ns, -1.0, done                                 # 每步 -1 / step cost -1

env=GridWorld(4)
print("状态数 / #states:", env.nS, "| 动作数 / #actions:", env.nA, "| 终点 / goal:", env.goal)
print("在状态0(左上角)向右走 / from state 0 move right(→):", env.step(0,1))
print("在状态0向上走(撞墙)/ move up(↑) into wall:", env.step(0,0))


**中文**：先理解**价值函数**——它是 RL 的核心概念。给定一个策略 $\pi$，**状态价值 $V^\pi(s)$** 回答"如果我从状态 $s$ 开始、一直按 $\pi$ 行动，未来能拿到多少累积(折扣)奖励？"。它把"这个位置好不好"量化成一个数。
**English**: First understand the **value function** — RL's central concept. Given a policy $\pi$, the **state value $V^\pi(s)$** answers "if I start in state $s$ and follow $\pi$ forever, how much cumulative (discounted) reward will I get?" It quantifies "how good is this position" as a single number.

**中文**：**贝尔曼期望方程**给出 $V^\pi$ 的递归定义——当前价值 = 即时奖励 + 折扣后的下一状态价值(对策略和转移求期望):
**English**: The **Bellman expectation equation** gives $V^\pi$ a recursive definition — current value = immediate reward + discounted next-state value (expected over policy and transitions):

$$V^\pi(s)=\sum_a\pi(a|s)\sum_{s'}P(s'|s,a)\big[R(s,a)+\gamma V^\pi(s')\big]$$

**中文**：GridWorld 的转移是**确定性**的(做动作一定到某格)，所以内层求和塌缩成单个下一状态。我们先给一个**随机策略**(每个方向各 25% 概率)，用**迭代法**求它的 $V^\pi$:反复用贝尔曼方程更新，直到收敛(这本质是解一个线性方程组的不动点迭代)。
**English**: GridWorld's transitions are **deterministic** (an action always lands in one cell), so the inner sum collapses to a single next-state. We start with a **random policy** (25% each direction) and compute its $V^\pi$ by **iteration**: repeatedly apply the Bellman equation until convergence (a fixed-point iteration for solving the linear system).


In [ ]:

# ============================================================
# 迭代法求随机策略的状态价值 V^π / evaluate V^π of the random policy by iteration
# ============================================================
gamma=0.9                                                     # 折扣因子 / discount
def evaluate_policy(env, policy, gamma=0.9, theta=1e-8):
    """policy[s,a] = 在状态 s 选动作 a 的概率 / probability of action a in state s."""
    V=np.zeros(env.nS)                                        # 初始化价值全 0 / init values
    while True:
        delta=0
        for state in range(env.nS):
            if state==env.goal: continue                      # 终点价值恒为0 / terminal value = 0
            v_new=0
            for a in range(env.nA):
                ns,r,done=env.step(state,a)                   # 确定性转移 / deterministic transition
                v_new += policy[state,a]*(r + gamma*V[ns])    # 贝尔曼期望方程一项 / one Bellman term
            delta=max(delta, abs(v_new-V[state]))             # 记录最大变化 / track max change
            V[state]=v_new
        if delta<theta: break                                 # 收敛就停 / stop when converged
    return V

random_policy=np.ones((env.nS,env.nA))/env.nA                 # 均匀随机策略 / uniform random policy
V_random=evaluate_policy(env, random_policy, gamma)
print("随机策略的状态价值 V^π (4x4 网格) / V of random policy:")
print(np.round(V_random.reshape(4,4),2))
print("\n解读:数越大(越接近0)=越好。终点(右下)=0; 离终点越远、价值越负(要走更多步、扣更多分)")
print("Reading: higher (closer to 0) = better. Goal(bottom-right)=0; farther cells are more negative")


**中文**：价值函数告诉我们"这个策略有多好"，但我们真正想要的是**最优策略**。这就要用**贝尔曼最优方程**——把"对策略求平均"换成"取最好的动作"(max):
**English**: The value function tells us "how good a policy is," but we really want the **optimal policy**. That needs the **Bellman optimality equation** — replacing "average over policy" with "take the best action" (max):

$$V^*(s)=\max_a\sum_{s'}P(s'|s,a)\big[R(s,a)+\gamma V^*(s')\big]$$

**中文**：直觉:最优状态价值 = 选那个"即时奖励 + 折扣后最优后继价值"最大的动作。一旦有了 $V^*$，最优策略就是在每个状态**贪心地**选让上式最大的动作。下面我们从当前随机策略出发，**贪心地**改进它一次(对每个状态选当前看起来最好的动作)，直观感受"策略改进"。
**English**: Intuition: the optimal state value = pick the action maximizing "immediate reward + discounted best successor value." Once we have $V^*$, the optimal policy is to act **greedily** with respect to it at each state. Below we take our random policy and **greedily improve** it once (each state picks the currently-best action) to feel "policy improvement."


In [ ]:

# ============================================================
# 贪心策略改进:基于 V 选每个状态的最佳动作 / greedy policy improvement from V
# ============================================================
def greedy_policy_from_V(env, V, gamma=0.9):
    policy=np.zeros((env.nS,env.nA))
    best_actions=np.zeros(env.nS,dtype=int)
    for state in range(env.nS):
        if state==env.goal: continue
        q=[]                                                  # 每个动作的 Q(s,a) / action values
        for a in range(env.nA):
            ns,r,done=env.step(state,a)
            q.append(r + gamma*V[ns])                         # Q(s,a)=即时奖励+折扣后继价值 / one-step lookahead
        best=int(np.argmax(q)); best_actions[state]=best
        policy[state,best]=1.0                                # 确定性贪心策略 / deterministic greedy
    return policy, best_actions

greedy_pi, best_a = greedy_policy_from_V(env, V_random, gamma)
V_greedy=evaluate_policy(env, greedy_pi, gamma)
print("对随机策略的V贪心改进后, 新策略的动作 / greedy actions (arrows):")
arrows=np.array([env.names[best_a[s]] if s!=env.goal else "G" for s in range(env.nS)]).reshape(4,4)
print(arrows)
print("\n改进后各状态价值 / V after one improvement:")
print(np.round(V_greedy.reshape(4,4),2))
print(f"随机策略平均价值 {V_random.mean():.2f} -> 改进后 {V_greedy.mean():.2f} (更接近0=更好)")


**中文**：可视化随机策略与改进后策略的价值图，直观看到"策略改进"让每个格子都更接近终点(价值升高)。
**English**: Visualize the value maps of the random and improved policies to see that "policy improvement" makes every cell closer to the goal (higher value).


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(15,4.5))
def draw(a, V, arrows_grid, title):
    G=V.reshape(4,4)
    im=a.imshow(G,cmap="RdYlGn",vmin=V.min(),vmax=0)
    for r in range(4):
        for c in range(4):
            s=r*4+c; txt=f"{G[r,c]:.1f}"
            if arrows_grid is not None and s!=env.goal: txt+=f"\n{arrows_grid[r,c]}"
            if s==env.goal: txt="GOAL"
            a.text(c,r,txt,ha="center",va="center",fontsize=10)
    a.set_title(title); a.set_xticks([]); a.set_yticks([]); return im
draw(ax[0], V_random, None, "随机策略 V^π / random policy")
im=draw(ax[1], V_greedy, arrows, "贪心改进后 V+策略 / greedy improved")
# ③ 折扣因子 gamma 的影响 / effect of discount gamma
for g in [0.5,0.9,0.99]:
    Vg=evaluate_policy(env, random_policy, g)
    ax[2].plot(Vg,"o-",label=f"γ={g}")
ax[2].set_title("折扣因子的影响 / effect of γ"); ax[2].set_xlabel("state"); ax[2].set_ylabel("V"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/rl01_viz.png",dpi=80); plt.show()
print("γ 越大越'远见'(价值差异被放大, 更看重远处终点)/ larger γ = more far-sighted")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **价值函数把"位置好坏"量化了**:随机策略下，离终点越远的格子价值越负(要瞎逛更久才到)。这就是 RL 的核心信号——不是"对/错"标签，而是"从这里出发能有多好"。
2. **贝尔曼方程是递归的心脏**:它把一个"看无穷远未来"的难题，拆成"即时奖励 + 下一步价值"的一步递归。整个 RL(动态规划、TD、Q-learning、乃至 DQN 的损失函数)都建立在这个递归上。
3. **一次贪心改进就明显变好**:仅仅"基于随机策略的价值、每个状态选当前最好的动作"，平均价值就从很负提升到接近最优。这预示了下一节**策略迭代/价值迭代**——反复"评估→改进"直到最优。
4. **γ 控制视野**:小 γ 短视(只顾眼前)，大 γ 远见(重视远处的终点)。选 γ 是 RL 建模的关键超参。

**English**:
1. **The value function quantifies "how good a position is"**: under the random policy, cells farther from the goal are more negative (more wandering to reach it). This is RL's core signal — not "right/wrong" labels, but "how good is it to start here."
2. **The Bellman equation is the recursive heart**: it decomposes an "infinite-horizon" problem into a one-step recursion "immediate reward + next-state value." All of RL (dynamic programming, TD, Q-learning, even DQN's loss) is built on this recursion.
3. **One greedy improvement already helps a lot**: merely "picking each state's best action w.r.t. the random policy's value" lifts the average value from very negative toward optimal. This foreshadows **policy/value iteration** next — repeatedly "evaluate → improve" until optimal.
4. **γ controls the horizon**: small γ is short-sighted (only immediate), large γ is far-sighted (values the distant goal). Choosing γ is a key modeling hyperparameter.

> 💼 **实战视角 / Practical angle**
> **中文**:把问题**建模成 MDP** 是应用 RL 的第一步、也最关键:想清楚"状态是什么、动作是什么、奖励怎么设、γ 取多少"。**奖励设计(reward shaping)** 是实战头号难题——奖励设错, 智能体会"钻空子"(reward hacking)。**马尔可夫性**常需靠"把历史塞进状态"来近似满足(如把最近几帧堆叠成状态)。面试金句:*"RL 就是在 MDP 里最大化折扣回报; 贝尔曼方程把长期价值递归成即时奖励加后继价值, 是所有 RL 算法的基础。"*
> **English**: **Modeling the problem as an MDP** is the first and most crucial step in applying RL: nail down "what are states, actions, rewards, and γ." **Reward design (reward shaping)** is the #1 practical challenge — a mis-specified reward invites "reward hacking." The **Markov property** is often approximated by "stuffing history into the state" (e.g. stacking recent frames). Interview line: *"RL maximizes discounted return in an MDP; the Bellman equation recursively expresses long-term value as immediate reward plus successor value — the foundation of every RL algorithm."*

---
### 小结 / Summary
- **中文**:RL=智能体在 MDP(S,A,P,R,γ)里试错最大化折扣回报; 核心是马尔可夫性。
- **English**: RL = an agent in an MDP (S,A,P,R,γ) learning by trial-and-error to maximize discounted return; core is the Markov property.
- **中文**:价值函数 V^π(s)/Q^π(s,a) 衡量"从某状态(动作)出发有多好"; 贝尔曼方程递归定义它。
- **English**: Value functions V^π(s)/Q^π(s,a) measure "how good it is to start from a state (action)"; the Bellman equation defines them recursively.
- **中文**:"评估当前策略价值→贪心改进"是找最优策略的基本循环(下节展开)。
- **English**: "Evaluate the current policy's value → greedily improve" is the basic loop for finding the optimal policy (next section).
